In [1]:
file_path = "../data/vto-4.67-0.6-T4-100k-torus-L.out"

with open(file_path, "r") as f:
    for i in range(20):
        print(f.readline().rstrip())


# ntime 4 20381 fixed
# Nso 16344 104277 253668 110490 100028 1.163670 3740
# v 11.022081 35.815585 1.015694 24.031789
# K0 4.670000 Delta 0.600000 K4 1.163670
# cid 1 cnt 20001 xid 7
Vto 1 11142 1470 1534 17867 3639 3479
Vto 2 17867 1340 1366 11400 5839 2766
Vto 3 11400 1208 1098 9605 3726 3717
Vto 4 9605 1276 1170 11142 3140 3740
# ntime 4 20382 fixed
# Nso 16313 104190 253630 110502 99922 1.163670 3687
# v 11.022052 35.815505 1.015694 24.031783
# K0 4.670000 Delta 0.600000 K4 1.163670
# cid 1 cnt 20002 xid 7
Vto 1 10711 1505 1423 17726 3505 3377
Vto 2 17726 1378 1352 11860 5779 2399
Vto 3 11860 1238 1126 9664 3871 3453
Vto 4 9664 1298 1260 10711 3158 3687
# ntime 4 20383 fixed
# Nso 16375 104592 254618 110934 100284 1.163670 3653


In [ ]:
# ntime 4 19485 fixed
# Nso 16388 101314 241376 104300 100080 1.204590 5238
# v 20.337005 67.349335 1.015692 23.951743
# K0 4.930000 Delta 0.600000 K4 1.204590
# cid 12 cnt 20001 xid 8
Vto 1 11212 504 498 10278 3667 4187
Vto 2 10278 481 493 13091 3362 4878
Vto 3 13091 597 569 15459 4283 5238
Vto 4 15459 551 527 11212 5076 3729
# ntime 4 19486 fixed
# Nso 16394 101448 241852 104532 100170 1.204590 5735
# v 20.337168 67.349891 1.015692 23.951776
# K0 4.930000 Delta 0.600000 K4 1.204590
# cid 12 cnt 20002 xid 8
Vto 1 10917 466 484 10387 3568 4166
Vto 2 10387 545 469 13549 3394 5198
Vto 3 13549 641 693 15232 4441 5735
Vto 4 15232 558 506 10917 4991 3667

In [5]:
def parse_k0(line: str) -> float:
    """
    K0 4.670000 Delta 0.600000 K4 1.163670
    """
    parts = line.split()
    k0_index = parts.index("K0")
    return float(parts[k0_index + 1])
    
def parse_nso(line: str):
    """
    Nso 16394 101448 241852 104532 100170 1.204590 5735
    """
    parts = line.split()
    idx = parts.index("Nso")
    values = parts[idx + 1:] ##what is that 1:
    return [float(v) for v in values]


def parse_v(line: str):
    """
    v 20.337005 67.349335 1.015692 23.951743
    """
    parts = line.split()
    idx = parts.index("v")
    return [float(v) for v in parts[idx + 1:]]

def parse_vto(line: str):
    """
    Zwraca tylko 6 liczb (bez indeksu czasu)
    """
    parts = line.split()
    # parts = ["Vto", "1", x1, x2, x3, x4, x5, x6]
    values = parts[2:]
    return [float(v) for v in values]
    

In [6]:
current_sample = {
    "K0": None,
    "Nso": None,
    "v": None,
    "Vto": []
}


current_sample["K0"] = parse_k0("# K0 4.930000 Delta 0.600000 K4 1.204590")
current_sample["Nso"] = parse_nso("# Nso 16388 101314 241376 104300 100080 1.204590 5238")
current_sample["v"] = parse_v("# v 20.337005 67.349335 1.015692 23.951743")

vto_lines = [
    "Vto 1 11212 504 498 10278 3667 4187",
    "Vto 2 10278 481 493 13091 3362 4878",
    "Vto 3 13091 597 569 15459 4283 5238",
    "Vto 4 15459 551 527 11212 5076 3729",
]

for line in vto_lines:
    current_sample["Vto"].append(parse_vto(line))

current_sample


{'K0': 4.93,
 'Nso': [16388.0, 101314.0, 241376.0, 104300.0, 100080.0, 1.20459, 5238.0],
 'v': [20.337005, 67.349335, 1.015692, 23.951743],
 'Vto': [[11212.0, 504.0, 498.0, 10278.0, 3667.0, 4187.0],
  [10278.0, 481.0, 493.0, 13091.0, 3362.0, 4878.0],
  [13091.0, 597.0, 569.0, 15459.0, 4283.0, 5238.0],
  [15459.0, 551.0, 527.0, 11212.0, 5076.0, 3729.0]]}

In [7]:
# teraz trzeba to wszystko spłaszczyć, dla modelu ml

def flatten_sample(sample: dict):
    """
    Zamienia próbkę CDT na 1D listę cech
    """
    features = []

    # global
    features.extend(sample["Nso"])
    features.extend(sample["v"])

    # local - kolejność czasowa
    for vto_t in sample["Vto"]:
        features.extend(vto_t)

    return features


In [8]:
flat = flatten_sample(current_sample)
len(flat), flat


(35,
 [16388.0,
  101314.0,
  241376.0,
  104300.0,
  100080.0,
  1.20459,
  5238.0,
  20.337005,
  67.349335,
  1.015692,
  23.951743,
  11212.0,
  504.0,
  498.0,
  10278.0,
  3667.0,
  4187.0,
  10278.0,
  481.0,
  493.0,
  13091.0,
  3362.0,
  4878.0,
  13091.0,
  597.0,
  569.0,
  15459.0,
  4283.0,
  5238.0,
  15459.0,
  551.0,
  527.0,
  11212.0,
  5076.0,
  3729.0])

In [9]:
# funkcja na permutację czasową, opisaną w artykule 
def time_permutations(sample: dict):
    """
    Zwraca 4 próbki z cykliczną permutacją czasu
    """
    perms = []
    V = sample["Vto"]

    for shift in range(4):
        permuted = {
            "K0": sample["K0"],
            "Nso": sample["Nso"],
            "v": sample["v"],
            "Vto": V[shift:] + V[:shift]
        }
        perms.append(permuted)

    return perms


In [10]:
perms = time_permutations(current_sample)

for i, s in enumerate(perms):
    print(f"perm {i+1}:")
    for row in s["Vto"]:
        print(row)
    print()


perm 1:
[11212.0, 504.0, 498.0, 10278.0, 3667.0, 4187.0]
[10278.0, 481.0, 493.0, 13091.0, 3362.0, 4878.0]
[13091.0, 597.0, 569.0, 15459.0, 4283.0, 5238.0]
[15459.0, 551.0, 527.0, 11212.0, 5076.0, 3729.0]

perm 2:
[10278.0, 481.0, 493.0, 13091.0, 3362.0, 4878.0]
[13091.0, 597.0, 569.0, 15459.0, 4283.0, 5238.0]
[15459.0, 551.0, 527.0, 11212.0, 5076.0, 3729.0]
[11212.0, 504.0, 498.0, 10278.0, 3667.0, 4187.0]

perm 3:
[13091.0, 597.0, 569.0, 15459.0, 4283.0, 5238.0]
[15459.0, 551.0, 527.0, 11212.0, 5076.0, 3729.0]
[11212.0, 504.0, 498.0, 10278.0, 3667.0, 4187.0]
[10278.0, 481.0, 493.0, 13091.0, 3362.0, 4878.0]

perm 4:
[15459.0, 551.0, 527.0, 11212.0, 5076.0, 3729.0]
[11212.0, 504.0, 498.0, 10278.0, 3667.0, 4187.0]
[10278.0, 481.0, 493.0, 13091.0, 3362.0, 4878.0]
[13091.0, 597.0, 569.0, 15459.0, 4283.0, 5238.0]



In [11]:
X = []
y = []

for perm in time_permutations(current_sample):
    X.append(flatten_sample(perm))
    y.append(1 if perm["K0"] == 4.93 else 0)


In [12]:
len(X), len(X[0])


(4, 35)